In [0]:
from pyspark.sql import functions as F

silver = spark.table(
    "workspace.retail_warehouse.silver_retail_transactions"
)

# Candidate business key
key_columns = [
    "invoice",
    "stockcode",
    "invoicedate",
    "customer_id"
]

# Check whether the candidate key is unique
duplicate_keys = (
    silver
    .groupBy(*key_columns)
    .count()
    .filter(F.col("count") > 1)
)

print("TOTAL SILVER RECORDS:", silver.count())
print("DUPLICATE BUSINESS KEYS:", duplicate_keys.count())

display(duplicate_keys.limit(10))

TOTAL SILVER RECORDS: 1033031
DUPLICATE BUSINESS KEYS: 11153


invoice,stockcode,invoicedate,customer_id,count
489529,22086,2009-12-01T11:51:00.000Z,17984,2
489536,21791,2009-12-01T12:13:00.000Z,16393,3
489536,85227,2009-12-01T12:13:00.000Z,16393,2
C489563,90092,2009-12-01T13:09:00.000Z,13526,2
489574,22088,2009-12-01T13:31:00.000Z,13097,2
489807,21016,2009-12-02T12:43:00.000Z,15581,2
489830,21356,2009-12-02T13:57:00.000Z,13623,2
489837,22147,2009-12-02T14:21:00.000Z,14396,2
489866,21313,2009-12-02T15:04:00.000Z,16200,2
489866,82578,2009-12-02T15:04:00.000Z,16200,2


In [0]:
from pyspark.sql import functions as F

silver = spark.table(
    "workspace.retail_warehouse.silver_retail_transactions"
)

business_columns = [
    "invoice",
    "stockcode",
    "description",
    "quantity",
    "invoicedate",
    "price",
    "customer_id",
    "country"
]

# Generate deterministic transaction-line ID
silver_with_id = silver.withColumn(
    "transaction_line_id",
    F.sha2(
        F.to_json(F.struct(*[F.col(c) for c in business_columns])),
        256
    )
)

# Validate uniqueness
total_records = silver_with_id.count()

unique_ids = silver_with_id.select(
    "transaction_line_id"
).distinct().count()

print("TOTAL RECORDS:", total_records)
print("UNIQUE TRANSACTION IDs:", unique_ids)
print("DUPLICATE IDs:", total_records - unique_ids)

display(
    silver_with_id.select(
        "transaction_line_id",
        "invoice",
        "stockcode",
        "quantity",
        "price"
    ).limit(10)
)

TOTAL RECORDS: 1033031
UNIQUE TRANSACTION IDs: 1033031
DUPLICATE IDs: 0


transaction_line_id,invoice,stockcode,quantity,price
1b56745d327f9723cc07a39c20b3f16ccd0c0347d57c89f18432b0d87c998ea4,489436,21755,18,5.45
1d0375cbcc07a094a15ec343c831f3b1b3a79bfc8c5483ef3fb1219e34d27e42,489436,22142,12,1.45
29d41e056d67d273f30970e2053aae5d17339e8029e4d2b06f18bbb1a0cf571b,489437,20703,3,4.25
3b6ba5d48e2d9c84c54ba95df7581546b081dd85801199256d30b5f727f65775,489438,84031A,56,3.00
b3bfed8b5706934f738e6eaba57a306e06f75d2cebd6ba71f231ae21558cadb0,489438,84519B,60,2.40
46d4d6442d88be6bd044904d5f90dcf0c97ce5169d5df6853aaf144bbe068a22,489438,85183A,60,2.40
e0dfa860d317727963a2ce9116a96f9cede1f862d6573ab1b7d5628a5bc65d1d,489439,22064,12,1.65
21dcfb1ed3bf5d3ff1a09cd8f8030e46746d2e0cab0c92d4eab2ce8ed80ffc01,489439,20749,2,7.95
5265728a9bec2424bb20f7fb774f4c1f1895bbfff3918326fddf1e00bd8ccb5d,489441,22321,12,0.72
ecce29301631115c0d43bead421700e6dd0308e2827ae2db14194cff27be9c1b,489442,22091,12,1.25


In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.retail_warehouse.silver_retail_transactions")

invoice_duplicates = (
    silver
    .groupBy("invoice")
    .count()
    .filter(F.col("count") > 1)
)

invoice_summary = (
    silver
    .groupBy("invoice")
    .agg(
        F.first("customer_id", ignorenulls=True).alias("customer_id"),
        F.first("country", ignorenulls=True).alias("country"),
        F.min("invoicedate").alias("invoice_date"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("total_amount").alias("invoice_total"),
        F.count("*").alias("line_count")
    )
)

summary_duplicates = (
    invoice_summary
    .groupBy("invoice")
    .count()
    .filter(F.col("count") > 1)
)

print("INVOICE SUMMARY RECORDS:", invoice_summary.count())
print("DUPLICATE INVOICE KEYS:", summary_duplicates.count())

INVOICE SUMMARY RECORDS: 53623
DUPLICATE INVOICE KEYS: 0


In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.retail_warehouse.silver_retail_transactions")

invoice_summary = (
    silver
    .groupBy("invoice")
    .agg(
        F.first("customer_id", ignorenulls=True).alias("customer_id"),
        F.first("country", ignorenulls=True).alias("country"),
        F.min("invoicedate").alias("invoice_date"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("total_amount").alias("invoice_total"),
        F.count("*").alias("line_count")
    )
    .withColumn("last_updated", F.current_timestamp())
)

target_table = "workspace.retail_warehouse.gold_invoice_summary"

(
    invoice_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

print("INITIAL INVOICE SUMMARY CREATED")
print("Records:", spark.table(target_table).count())

INITIAL INVOICE SUMMARY CREATED
Records: 53623


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.retail_warehouse.gold_invoice_summary"
)

# Pick one existing invoice and simulate an updated total.
existing = (
    spark.table("workspace.retail_warehouse.gold_invoice_summary")
    .limit(1)
    .collect()[0]
)

existing_invoice = existing["invoice"]
updated_total = float(existing["invoice_total"]) + 100

# Create a small test source containing:
# 1. One updated existing invoice
# 2. One new invoice
test_source = spark.createDataFrame(
    [
        (
            existing_invoice,
            existing["customer_id"],
            existing["country"],
            existing["invoice_date"],
            existing["total_quantity"],
            updated_total,
            existing["line_count"]
        ),
        (
            "TEST_INCREMENTAL_001",
            999999,
            "Test Country",
            existing["invoice_date"],
            5,
            250.00,
            2
        )
    ],
    """
    invoice string,
    customer_id long,
    country string,
    invoice_date timestamp,
    total_quantity long,
    invoice_total double,
    line_count long
    """
).withColumn("last_updated", F.current_timestamp())

test_source.createOrReplaceTempView("invoice_merge_test_source")

spark.sql("""
MERGE INTO workspace.retail_warehouse.gold_invoice_summary AS target
USING invoice_merge_test_source AS source
ON target.invoice = source.invoice

WHEN MATCHED THEN UPDATE SET
    target.customer_id = source.customer_id,
    target.country = source.country,
    target.invoice_date = source.invoice_date,
    target.total_quantity = source.total_quantity,
    target.invoice_total = source.invoice_total,
    target.line_count = source.line_count,
    target.last_updated = source.last_updated

WHEN NOT MATCHED THEN INSERT *
""")

print("MERGE TEST COMPLETED")
print("Updated invoice total:",
      spark.table("workspace.retail_warehouse.gold_invoice_summary")
      .filter(F.col("invoice") == existing_invoice)
      .select("invoice_total").first()[0])

print("Inserted test invoice:",
      spark.table("workspace.retail_warehouse.gold_invoice_summary")
      .filter(F.col("invoice") == "TEST_INCREMENTAL_001")
      .count())

MERGE TEST COMPLETED
Updated invoice total: 1096.10
Inserted test invoice: 1


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

table_name = "workspace.retail_warehouse.gold_invoice_summary"

# Restore the original invoice total from the Silver source.
silver = spark.table("workspace.retail_warehouse.silver_retail_transactions")

original = (
    silver
    .filter(F.col("invoice") == existing_invoice)
    .agg(F.sum("total_amount").alias("original_total"))
    .first()["original_total"]
)

spark.sql(f"""
UPDATE {table_name}
SET invoice_total = {float(original)}
WHERE invoice = '{existing_invoice}'
""")

# Remove synthetic test invoice.
spark.sql(f"""
DELETE FROM {table_name}
WHERE invoice = 'TEST_INCREMENTAL_001'
""")

print("TEST CLEANUP COMPLETED")
print("Restored invoice total:", original)
print(
    "Synthetic test invoices remaining:",
    spark.table(table_name)
    .filter(F.col("invoice") == "TEST_INCREMENTAL_001")
    .count()
)
print("Final records:", spark.table(table_name).count())

TEST CLEANUP COMPLETED
Restored invoice total: 996.10
Synthetic test invoices remaining: 0
Final records: 53623


In [0]:
silver = spark.table("workspace.retail_warehouse.silver_retail_transactions")

display(
    silver
    .groupBy("source_file")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("invoice").alias("invoice_count")
    )
)

source_file,record_count,invoice_count
dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,518376,28807
dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2010-2011.csv,514655,24855


In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.retail_warehouse.silver_retail_transactions")

batch_1 = silver.filter(
    F.col("source_file").endswith("year_2009-2010.csv")
)

batch_2 = silver.filter(
    F.col("source_file").endswith("year_2010-2011.csv")
)

print("BATCH 1 RECORDS:", batch_1.count())
print("BATCH 2 RECORDS:", batch_2.count())
print("TOTAL RECORDS:", batch_1.count() + batch_2.count())

BATCH 1 RECORDS: 518376
BATCH 2 RECORDS: 514655
TOTAL RECORDS: 1033031


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

silver = spark.table(
    "workspace.retail_warehouse.silver_retail_transactions"
)

batch_1 = silver.filter(
    F.col("source_file").endswith("year_2009-2010.csv")
)

batch_2 = silver.filter(
    F.col("source_file").endswith("year_2010-2011.csv")
)

def create_invoice_summary(df):
    return (
        df.groupBy("invoice")
        .agg(
            F.first("customer_id", ignorenulls=True).alias("customer_id"),
            F.first("country", ignorenulls=True).alias("country"),
            F.min("invoicedate").alias("invoice_date"),
            F.sum("quantity").alias("total_quantity"),
            F.sum("total_amount").alias("invoice_total"),
            F.count("*").alias("line_count")
        )
        .withColumn("last_updated", F.current_timestamp())
    )

target_table = "workspace.retail_warehouse.gold_invoice_summary"

# Initial load from Batch 1
initial_summary = create_invoice_summary(batch_1)

(
    initial_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

print("INITIAL BATCH INVOICES:", initial_summary.count())

# Prepare incremental Batch 2
incremental_summary = create_invoice_summary(batch_2)

# Merge Batch 2 into the existing Delta table
target = DeltaTable.forName(spark, target_table)

(
    target.alias("target")
    .merge(
        incremental_summary.alias("source"),
        "target.invoice = source.invoice"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("INCREMENTAL BATCH INVOICES:", incremental_summary.count())
print("FINAL INVOICE SUMMARY:", spark.table(target_table).count())

INITIAL BATCH INVOICES: 28807
INCREMENTAL BATCH INVOICES: 24855
FINAL INVOICE SUMMARY: 53623


In [0]:
final_summary = spark.table(
    "workspace.retail_warehouse.gold_invoice_summary"
)

print("FINAL VALIDATION")
print("Total invoices:", final_summary.count())
print(
    "Duplicate invoice keys:",
    final_summary.groupBy("invoice")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print("Total revenue:", final_summary.agg(
    F.sum("invoice_total")
).first()[0])

display(final_summary.orderBy("invoice").limit(10))

FINAL VALIDATION
Total invoices: 53623
Duplicate invoice keys: 0
Total revenue: 18992828.21


invoice,customer_id,country,invoice_date,total_quantity,invoice_total,line_count,last_updated
489434,13085,United Kingdom,2009-12-01T07:45:00.000Z,166,505.30,8,2026-11-08T11:08:47.324Z
489435,13085,United Kingdom,2009-12-01T07:46:00.000Z,60,145.80,4,2026-11-08T11:08:47.324Z
489436,13078,United Kingdom,2009-12-01T09:06:00.000Z,193,630.33,19,2026-11-08T11:08:47.324Z
489437,15362,United Kingdom,2009-12-01T09:08:00.000Z,145,310.75,23,2026-11-08T11:08:47.324Z
489438,18102,United Kingdom,2009-12-01T09:24:00.000Z,826,2286.24,17,2026-11-08T11:08:47.324Z
489439,12682,France,2009-12-01T09:28:00.000Z,219,426.30,19,2026-11-08T11:08:47.324Z
489440,18087,United Kingdom,2009-12-01T09:43:00.000Z,16,50.40,2,2026-11-08T11:08:47.324Z
489441,18087,United Kingdom,2009-12-01T09:44:00.000Z,102,344.34,4,2026-11-08T11:08:47.324Z
489442,13635,United Kingdom,2009-12-01T09:46:00.000Z,275,382.37,23,2026-11-08T11:08:47.324Z
489443,14110,United Kingdom,2009-12-01T09:50:00.000Z,120,285.06,7,2026-11-08T11:08:47.324Z


In [0]:
from pyspark.sql import functions as F

silver = spark.table(
    "workspace.retail_warehouse.silver_retail_transactions"
)

# Aggregate ALL Silver records together.
# This ensures invoices appearing in both files are combined.
corrected_summary = (
    silver
    .groupBy("invoice")
    .agg(
        F.first("customer_id", ignorenulls=True).alias("customer_id"),
        F.first("country", ignorenulls=True).alias("country"),
        F.min("invoicedate").alias("invoice_date"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("total_amount").alias("invoice_total"),
        F.count("*").alias("line_count")
    )
    .withColumn("last_updated", F.current_timestamp())
)

target_table = "workspace.retail_warehouse.gold_invoice_summary"

(
    corrected_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("CORRECTED INVOICE SUMMARY CREATED")
print("Invoice records:", corrected_summary.count())

CORRECTED INVOICE SUMMARY CREATED
Invoice records: 53623


In [0]:
silver_revenue = (
    silver.agg(F.sum("total_amount")).first()[0]
)

gold_revenue = (
    spark.table(target_table)
    .agg(F.sum("invoice_total"))
    .first()[0]
)

print("SILVER REVENUE:", silver_revenue)
print("GOLD REVENUE:", gold_revenue)
print("DIFFERENCE:", silver_revenue - gold_revenue)

SILVER REVENUE: 19014209.82
GOLD REVENUE: 19014209.82
DIFFERENCE: 0.00


In [0]:
from pyspark.sql import functions as F

silver = spark.table(
    "workspace.retail_warehouse.silver_retail_transactions"
)

gold = spark.table(
    "workspace.retail_warehouse.gold_invoice_summary"
)

checks = {
    "Silver table is not empty": silver.count() > 0,

    "Gold table is not empty": gold.count() > 0,

    "No duplicate invoice keys": (
        gold.groupBy("invoice")
        .count()
        .filter(F.col("count") > 1)
        .count() == 0
    ),

    "No null invoice keys": (
        gold.filter(F.col("invoice").isNull()).count() == 0
    ),

    "Revenue reconciliation": (
        abs(
            float(silver.agg(F.sum("total_amount")).first()[0])
            - float(gold.agg(F.sum("invoice_total")).first()[0])
        ) < 0.01
    ),

    "Invoice counts reconcile": (
        gold.count() == silver.select("invoice").distinct().count()
    )
}

print("=" * 55)
print("DATA QUALITY VALIDATION")
print("=" * 55)

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print("=" * 55)
print("TOTAL CHECKS:", len(checks))
print("PASSED:", sum(checks.values()))
print("FAILED:", len(checks) - sum(checks.values()))

DATA QUALITY VALIDATION
PASS: Silver table is not empty
PASS: Gold table is not empty
PASS: No duplicate invoice keys
PASS: No null invoice keys
PASS: Revenue reconciliation
PASS: Invoice counts reconcile
TOTAL CHECKS: 6
PASSED: 6
FAILED: 0
